<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-40.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 40 - Preparación DUAL + Guía de referencia

**Módulos:** Sistemas de Big Data (SBD) + Big Data Aplicado (BDA)

---

## Objetivo

Este notebook es una **guía de referencia rápida** de todas las herramientas y técnicas aprendidas durante el periodo presencial. Úsalo como cheatsheet durante el periodo DUAL.

- **SBD (5074):** RA1 Técnicas de análisis, RA2 Cuadros de mando, RA3 Gestión y almacenamiento
- **BDA (5075):** RA1 Gestión de soluciones, RA2 Ecosistemas Big Data, RA3 Integridad, RA4 Monitorización

---

## 1. Complejidad algorítmica (SBD-RA1)

In [ ]:
import math
import time

n = 1_000_000

complejidades = {
    'O(1)':       1,
    'O(log n)':   math.log2(n),
    'O(n)':       n,
    'O(n log n)': n * math.log2(n),
    'O(n²)':      n ** 2,
}

print(f"Operaciones necesarias para n = {n:,}:\n")
for nombre, ops in complejidades.items():
    print(f"  {nombre:12s}: {ops:>20,.0f} operaciones")

print("\n💡 Recuerda: en Big Data, O(n²) es inviable.")
print("   Busca siempre O(n) o O(n log n) como máximo.")

---

## 2. Pandas - Operaciones esenciales (SBD-RA1/RA3)

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.DataFrame({
    'producto': np.random.choice(['A', 'B', 'C', 'D'], 100),
    'cantidad': np.random.randint(1, 20, 100),
    'precio': np.round(np.random.uniform(10, 100, 100), 2),
    'fecha': pd.date_range('2025-01-01', periods=100, freq='D')
})

# Filtrar
filtrado = df[df['cantidad'] > 10]

# Agrupar y agregar
resumen = df.groupby('producto').agg(
    total_qty=('cantidad', 'sum'),
    precio_medio=('precio', 'mean'),
    num_ventas=('producto', 'count')
).round(2)

# Merge de DataFrames
info_productos = pd.DataFrame({
    'producto': ['A', 'B', 'C', 'D'],
    'categoria': ['Electrónica', 'Accesorios', 'Electrónica', 'Software']
})
df_completo = pd.merge(df, info_productos, on='producto')

# Pivot table
pivot = df.pivot_table(values='cantidad', index=df['fecha'].dt.month,
                       columns='producto', aggfunc='sum', fill_value=0)

print("=== RESUMEN POR PRODUCTO ===")
print(resumen)
print(f"\n=== PIVOT (cantidad por mes × producto) ===")
print(pivot)

---

## 3. SQL y bases de datos (SBD-RA3 / BDA-RA2)

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
df.to_sql('ventas', conn, if_exists='replace', index=False)

consultas = {
    'Filtro + agregación': '''
        SELECT producto, SUM(cantidad) as total, AVG(precio) as precio_medio
        FROM ventas WHERE cantidad > 5
        GROUP BY producto ORDER BY total DESC
    ''',
    'Subquery': '''
        SELECT * FROM ventas
        WHERE precio > (SELECT AVG(precio) FROM ventas)
        LIMIT 5
    ''',
    'Window function (simulada)': '''
        SELECT producto, cantidad, precio,
               SUM(cantidad) OVER (PARTITION BY producto) as total_producto
        FROM ventas LIMIT 10
    '''
}

for titulo, query in consultas.items():
    print(f"\n=== {titulo} ===")
    print(pd.read_sql(query, conn).to_string(index=False))

conn.close()

---

## 4. MapReduce en Python (BDA-RA2)

In [ ]:
from collections import defaultdict
from functools import reduce

datos = ['laptop mouse laptop teclado monitor laptop mouse teclado',
         'monitor laptop auriculares mouse laptop teclado mouse',
         'laptop laptop monitor auriculares teclado mouse laptop']

# MAP
def mapper(linea):
    return [(palabra, 1) for palabra in linea.split()]

mapped = []
for linea in datos:
    mapped.extend(mapper(linea))

# SHUFFLE
shuffled = defaultdict(list)
for key, val in mapped:
    shuffled[key].append(val)

# REDUCE
resultado = {key: sum(values) for key, values in shuffled.items()}

print("=== WORD COUNT (MapReduce) ===")
for palabra, conteo in sorted(resultado.items(), key=lambda x: -x[1]):
    print(f"  {palabra:15s} → {conteo}")

---

## 5. Calidad e integridad de datos (BDA-RA3)

In [ ]:
import hashlib

# Calidad: detectar problemas
df_sucio = pd.DataFrame({
    'nombre': ['Ana', None, 'Luis', 'Ana', 'María', ''],
    'edad': [25, 30, -5, 25, 150, 40],
    'email': ['ana@mail.com', 'bob@mail.com', 'luis@mail', 'ana@mail.com', 'maria@mail.com', 'test@mail.com']
})

print("=== INFORME DE CALIDAD ===")
print(f"Registros totales: {len(df_sucio)}")
print(f"Nulos: {df_sucio.isnull().sum().to_dict()}")
print(f"Vacíos: {(df_sucio == '').sum().to_dict()}")
print(f"Duplicados: {df_sucio.duplicated().sum()}")
print(f"Edades fuera de rango (0-120): {((df_sucio['edad'] < 0) | (df_sucio['edad'] > 120)).sum()}")

# Integridad: checksums
print("\n=== CHECKSUMS ===")
datos_ejemplo = b"Datos importantes que no deben corromperse"
print(f"MD5:    {hashlib.md5(datos_ejemplo).hexdigest()}")
print(f"SHA256: {hashlib.sha256(datos_ejemplo).hexdigest()}")

# Verificación
datos_ok = b"Datos importantes que no deben corromperse"
datos_corrupto = b"Datos importantes que no deben corromperse!"
hash_original = hashlib.sha256(datos_ejemplo).hexdigest()

print(f"\nVerificación OK:      {hashlib.sha256(datos_ok).hexdigest() == hash_original}")
print(f"Verificación corrupto: {hashlib.sha256(datos_corrupto).hexdigest() == hash_original}")

---

## 6. Monitorización y métricas (BDA-RA4)

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import random

random.seed(42)
horas = 48
timestamps = [datetime(2026, 3, 10) + timedelta(hours=i) for i in range(horas)]

# Simular métricas de clúster
cpu = [40 + 20*math.sin(i/6) + random.gauss(0, 5) for i in range(horas)]
memoria = [60 + 10*math.sin(i/8) + random.gauss(0, 3) for i in range(horas)]
disco_io = [random.gauss(500, 100) for _ in range(horas)]

# Inyectar anomalía
cpu[30] = 95
cpu[31] = 98
memoria[30] = 92
memoria[31] = 95

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(timestamps, cpu, 'b-', linewidth=1)
axes[0].axhline(y=90, color='r', linestyle='--', alpha=0.7, label='Umbral alerta')
axes[0].fill_between(timestamps, cpu, 90, where=[c > 90 for c in cpu],
                     color='red', alpha=0.3)
axes[0].set_ylabel('CPU (%)')
axes[0].set_title('Monitorización del clúster (48h)')
axes[0].legend()

axes[1].plot(timestamps, memoria, 'g-', linewidth=1)
axes[1].axhline(y=90, color='r', linestyle='--', alpha=0.7)
axes[1].fill_between(timestamps, memoria, 90, where=[m > 90 for m in memoria],
                     color='red', alpha=0.3)
axes[1].set_ylabel('Memoria (%)')

axes[2].bar(timestamps, disco_io, width=0.03, color='purple', alpha=0.7)
axes[2].set_ylabel('Disco I/O (MB/s)')
axes[2].set_xlabel('Tiempo')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Detección de alertas
print("\n=== ALERTAS DETECTADAS ===")
for i, t in enumerate(timestamps):
    alertas = []
    if cpu[i] > 90: alertas.append(f"CPU={cpu[i]:.1f}%")
    if memoria[i] > 90: alertas.append(f"MEM={memoria[i]:.1f}%")
    if alertas:
        print(f"  ⚠️  {t.strftime('%Y-%m-%d %H:%M')}: {', '.join(alertas)}")

---

## 7. Checklist para el periodo DUAL

### SBD - RA4: Visualización de datos

- [ ] Portfolio de visualizaciones realizadas en la empresa
- [ ] Informe de herramientas utilizadas
- [ ] Proyecto final de visualización con datos reales

### BDA - RA5: Inteligencia de Negocios (BI)

- [ ] Análisis de datos reales aplicando el proceso KDD
- [ ] Informe de implantación de modelo BI
- [ ] Presentación de insights y decisiones de negocio
- [ ] Evaluación del impacto del análisis Big Data

### Herramientas recomendadas para DUAL

| Herramienta | Uso | Recurso |
|-------------|-----|--------|
| Python/Pandas | Análisis de datos | [pandas.pydata.org](https://pandas.pydata.org/) |
| Matplotlib/Seaborn | Visualización estática | [matplotlib.org](https://matplotlib.org/) |
| Plotly | Visualización interactiva | [plotly.com](https://plotly.com/) |
| Power BI | Dashboards empresariales | [powerbi.microsoft.com](https://powerbi.microsoft.com/) |
| Tableau | Visualización avanzada | [tableau.com](https://www.tableau.com/) |
| Google Colab | Entorno de desarrollo | [colab.research.google.com](https://colab.research.google.com/) |